# Get racmo downscaled fields and upload to scratch bucket

This downloads all the netcdfs supplied ot me by Brice Noel in an email to j.kingslake@columbia.edu on May 27 2026 and puts them in my cryocloud stratch bucket. 

Main code developed using Chatgpt.

- J.Kingslake, May 27, 2026

In [12]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=2,
    memory_limit="8GB",
)

client = Client(cluster)

client

/srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38849 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/proxy/38849/status,
Dashboard: /user/jkingslake/proxy/38849/status,Workers: 4
Total threads: 8,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43313,Workers: 4
Dashboard: /user/jkingslake/proxy/38849/status,Total threads: 8
Started: Just now,Total memory: 29.80 GiB
Comm: tcp://127.0.0.1:35155,Total threads: 2
Dashboard: /user/jkingslake/proxy/44035/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:40967,


2026-05-27 19:36:52,239 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:40917'.
2026-05-27 19:36:52,241 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:45691'.


In [ ]:
client.shutdown()

In [3]:
import requests
import xarray as xr
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from tqdm.auto import tqdm

import s3fs

fs = s3fs.S3FileSystem()

BASE_URL = "http://climato.be/ftp/climato/bnoel/Share-public/.Kingslake/Daily-2km/"

# DATASETS = ["snowmelt", "ff10m", "precip", "t2m"]
DATASETS = ["precip"]

S3_BASE = "s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr"

TMP_DIR = Path("tmp_nc")
TMP_DIR.mkdir(exist_ok=True)


def list_nc_files(url):
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    return sorted(
        a["href"]
        for a in soup.find_all("a", href=True)
        if a["href"].endswith(".nc")
    )


for dataset in DATASETS:
    dataset_url = urljoin(BASE_URL, dataset + "/")
    files = list_nc_files(dataset_url)

    print(f"{dataset}: {len(files)} files")

    for filename in tqdm(files):
        source_url = urljoin(dataset_url, filename)
        local_nc = TMP_DIR / filename

        zarr_name = Path(filename).stem + ".zarr"
        zarr_path = f"{S3_BASE}/{dataset}/{zarr_name}"

        if fs.exists(zarr_path + "/.zmetadata"):

            print(f"Skipping existing zarr {zarr_path}")
        
            continue
        
        if local_nc.exists():
            print(f"Using existing local file {local_nc}")
        else:
            print(f"Downloading {filename}")
        
            r = requests.get(source_url)
            r.raise_for_status()

            local_nc.write_bytes(r.content)

        print(f"Opening {filename}")
        ds = xr.open_dataset(local_nc, engine="h5netcdf", chunks="auto")
        
        try:
            print(f"Writing {zarr_path}")
            ds.to_zarr(
                zarr_path,
                mode="w",
                consolidated=True,
                zarr_format=2,
            )
        except Exception as e:

            print("failed .load()",local_nc, e)
            continue
        
        ds.close()
        local_nc.unlink()

print("Done")

precip: 188 files


  0%|          | 0/188 [00:00<?, ?it/s]

Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1979_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1979_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1979_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1980_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1980_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/precip/precip.1980_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr
Skipping existing zarr s3:/